In [1]:
# @title NurseSchedulingProblem Class
import numpy as np

class NurseSchedulingProblem:
    """This class encapsulates the Nurse Scheduling problem
    """

    def __init__(self, hardConstraintPenalty):
        """
        :param hardConstraintPenalty: the penalty factor for a hard-constraint violation
        """
        self.hardConstraintPenalty = hardConstraintPenalty

        # list of nurses:
        self.nurses = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']

        # nurses' respective shift preferences - morning, evening, night:
        self.shiftPreference = [[1, 0, 0], [1, 1, 0], [0, 0, 1], [0, 1, 0], [0, 0, 1], [1, 1, 1], [0, 1, 1], [1, 1, 1]]

        # min and max number of nurses allowed for each shift - morning, evening, night:
        self.shiftMin = [2, 2, 1]
        self.shiftMax = [3, 4, 2]

        # max shifts per week allowed for each nurse
        self.maxShiftsPerWeek = 5

        # number of weeks we create a schedule for:
        self.weeks = 1

        # useful values:
        self.shiftPerDay = len(self.shiftMin)
        self.shiftsPerWeek = 7 * self.shiftPerDay

    def __len__(self):
        """
        :return: the number of shifts in the schedule
        """
        return len(self.nurses) * self.shiftsPerWeek * self.weeks


    def getCost(self, schedule):
        """
        Calculates the total cost of the various violations in the given schedule
        ...
        :param schedule: a list of binary values describing the given schedule
        :return: the calculated cost
        """

        if len(schedule) != self.__len__():
            raise ValueError("size of schedule list should be equal to ", self.__len__())

        # convert entire schedule into a dictionary with a separate schedule for each nurse:
        nurseShiftsDict = self.getNurseShifts(schedule)

        # count the various violations:
        consecutiveShiftViolations = self.countConsecutiveShiftViolations(nurseShiftsDict)
        shiftsPerWeekViolations = self.countShiftsPerWeekViolations(nurseShiftsDict)[1]
        nursesPerShiftViolations = self.countNursesPerShiftViolations(nurseShiftsDict)[1]
        shiftPreferenceViolations = self.countShiftPreferenceViolations(nurseShiftsDict)

        # calculate the cost of the violations:
        hardContstraintViolations = consecutiveShiftViolations + nursesPerShiftViolations + shiftsPerWeekViolations
        softContstraintViolations = shiftPreferenceViolations

        return self.hardConstraintPenalty * hardContstraintViolations + softContstraintViolations

    def getNurseShifts(self, schedule):
        """
        Converts the entire schedule into a dictionary with a separate schedule for each nurse
        :param schedule: a list of binary values describing the given schedule
        :return: a dictionary with each nurse as a key and the corresponding shifts as the value
        """
        shiftsPerNurse = self.__len__() // len(self.nurses)
        nurseShiftsDict = {}
        shiftIndex = 0

        for nurse in self.nurses:
            nurseShiftsDict[nurse] = schedule[shiftIndex:shiftIndex + shiftsPerNurse]
            shiftIndex += shiftsPerNurse

        return nurseShiftsDict

    def countConsecutiveShiftViolations(self, nurseShiftsDict):
        """
        Counts the consecutive shift violations in the schedule
        :param nurseShiftsDict: a dictionary with a separate schedule for each nurse
        :return: count of violations found
        """
        violations = 0
        # iterate over the shifts of each nurse:
        for nurseShifts in nurseShiftsDict.values():
            # look for two cosecutive '1's:
            for shift1, shift2 in zip(nurseShifts, nurseShifts[1:]):
                if shift1 == 1 and shift2 == 1:
                    violations += 1
        return violations

    def countShiftsPerWeekViolations(self, nurseShiftsDict):
        """
        Counts the max-shifts-per-week violations in the schedule
        :param nurseShiftsDict: a dictionary with a separate schedule for each nurse
        :return: count of violations found
        """
        violations = 0
        weeklyShiftsList = []
        # iterate over the shifts of each nurse:
        for nurseShifts in nurseShiftsDict.values():  # all shifts of a single nurse
            # iterate over the shifts of each weeks:
            for i in range(0, self.weeks * self.shiftsPerWeek, self.shiftsPerWeek):
                # count all the '1's over the week:
                weeklyShifts = sum(nurseShifts[i:i + self.shiftsPerWeek])
                weeklyShiftsList.append(weeklyShifts)
                if weeklyShifts > self.maxShiftsPerWeek:
                    violations += weeklyShifts - self.maxShiftsPerWeek

        return weeklyShiftsList, violations

    def countNursesPerShiftViolations(self, nurseShiftsDict):
        """
        Counts the number-of-nurses-per-shift violations in the schedule
        :param nurseShiftsDict: a dictionary with a separate schedule for each nurse
        :return: count of violations found
        """
        # sum the shifts over all nurses:
        totalPerShiftList = [sum(shift) for shift in zip(*nurseShiftsDict.values())]

        violations = 0
        # iterate over all shifts and count violations:
        for shiftIndex, numOfNurses in enumerate(totalPerShiftList):
            dailyShiftIndex = shiftIndex % self.shiftPerDay  # -> 0, 1, or 2 for the 3 shifts per day
            if (numOfNurses > self.shiftMax[dailyShiftIndex]):
                violations += numOfNurses - self.shiftMax[dailyShiftIndex]
            elif (numOfNurses < self.shiftMin[dailyShiftIndex]):
                violations += self.shiftMin[dailyShiftIndex] - numOfNurses

        return totalPerShiftList, violations

    def countShiftPreferenceViolations(self, nurseShiftsDict):
        """
        Counts the nurse-preferences violations in the schedule
        :param nurseShiftsDict: a dictionary with a separate schedule for each nurse
        :return: count of violations found
        """
        violations = 0
        for nurseIndex, shiftPreference in enumerate(self.shiftPreference):
            # duplicate the shift-preference over the days of the period
            preference = shiftPreference * (self.shiftsPerWeek // self.shiftPerDay)
            # iterate over the shifts and compare to preferences:
            shifts = nurseShiftsDict[self.nurses[nurseIndex]]
            for pref, shift in zip(preference, shifts):
                if pref == 0 and shift == 1:
                    violations += 1

        return violations

    def printScheduleInfo(self, schedule):
        """
        Prints the schedule and violations details
        :param schedule: a list of binary values describing the given schedule
        """
        nurseShiftsDict = self.getNurseShifts(schedule)

        print("Schedule for each nurse:")
        for nurse in nurseShiftsDict:  # all shifts of a single nurse
            print(nurse, ":", nurseShiftsDict[nurse])

        print("consecutive shift violations = ", self.countConsecutiveShiftViolations(nurseShiftsDict))
        print()

        weeklyShiftsList, violations = self.countShiftsPerWeekViolations(nurseShiftsDict)
        print("weekly Shifts = ", weeklyShiftsList)
        print("Shifts Per Week Violations = ", violations)
        print()

        totalPerShiftList, violations = self.countNursesPerShiftViolations(nurseShiftsDict)
        print("Nurses Per Shift = ", totalPerShiftList)
        print("Nurses Per Shift Violations = ", violations)
        print()

        shiftPreferenceViolations = self.countShiftPreferenceViolations(nurseShiftsDict)
        print("Shift Preference Violations = ", shiftPreferenceViolations)
        print()

In [2]:
!pip install deap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.0/136.0 kB 4.4 MB/s eta 0:00:00


# ============================================================
# 🧬 Genetic Algorithms Workshop: Nurse Scheduling Problem
# ============================================================

Welcome to your Genetic Algorithms exercise!

🎯 **Goal**
You’ll learn how to implement and tune a Genetic Algorithm (GA)
to solve a real-world optimization problem — the *Nurse Scheduling Problem*.

The task is to automatically assign nurses to shifts, while respecting:
- **Hard constraints** (rules that cannot be broken)
- **Soft constraints** (preferences that should be met as much as possible)

By the end, you’ll:
- Understand how DEAP simplifies evolutionary algorithms.
- Implement your own GA operators.
- Visualize the evolution of solutions over generations.

📋 **Instructions**
1. Run each cell in order.
2. Fill in the parts marked with `### TODO` — these are exercises.
3. You can test and re-run cells as needed.
4. Use the comments and hints provided to guide your thinking.

💡 **Tips**
- Read the DEAP documentation if you’re unsure about a function: https://deap.readthedocs.io
- Experiment! Try changing population size, mutation rate, or generations to see how it affects results.

In [4]:
# ============================================================
# Imports
# ============================================================

from deap import base, creator, tools, algorithms

import random
import numpy
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:

# ============================================================
# Problem Constants
# ============================================================

HARD_CONSTRAINT_PENALTY = 10  # penalty factor for constraint violations

# Genetic Algorithm constants
POPULATION_SIZE = 300
P_CROSSOVER = 0.9     # probability of crossover
P_MUTATION = 0.1      # probability of mutation
MAX_GENERATIONS = 200
HALL_OF_FAME_SIZE = 30

# Set random seed for reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

In [6]:
# ============================================================
# Problem Setup: Nurse Scheduling
# ============================================================

### Assume NurseSchedulingProblem is defined elsewhere in your notebook

toolbox = base.Toolbox()

# Create the nurse scheduling problem instance:
### TODO: create a problem instance using the HARD_CONSTRAINT_PENALTY

# Example: nsp = NurseSchedulingProblem(HARD_CONSTRAINT_PENALTY)

nsp = None  # TODO

In [ ]:
# ============================================================
# Create Genetic Representation
# ============================================================

### Step 1. Define Fitness and Individual classes

# Hint: We want to minimize the scheduling cost.

### TODO: Use DEAP's creator to define:
# - FitnessMin (inherits from base.Fitness)
# - Individual (inherits from list, uses FitnessMin as fitness)

### TODO

In [ ]:
# ============================================================
# Register Initialization Operators
# ============================================================

# Create a binary gene generator (0 or 1)
### TODO: register a generator that randomly returns 0 or 1
# toolbox.register("zeroOrOne", ...)

### HINT: use random.randint(0, 1)

### TODO

# Create an individual (list of genes)
### TODO: register individual creation
# toolbox.register("individualCreator", ...)

### TODO

# Create a population of individuals
### TODO: register population creation
# toolbox.register("populationCreator", ...)

### TODO

In [ ]:
# ============================================================
# Define the Fitness Function
# ============================================================

### Step 1: Define getCost()
# This function should compute and return the total cost of an individual.
# Hint: use the problem instance’s `getCost()` method.

### TODO
def getCost(individual):
    pass

### Step 2: Register evaluation function
### TODO
# toolbox.register("evaluate", ...)


In [ ]:
# ============================================================
# Define Genetic Operators
# ============================================================

### TODO: Register selection, crossover, and mutation operators
# Selection → Tournament selection
# Crossover → Two-point crossover
# Mutation → Flip bit mutation with probability 1/len(nsp)

# Example:
# toolbox.register("select", ...)
# toolbox.register("mate", ...)
# toolbox.register("mutate", ...)

### TODO


In [ ]:
# ============================================================
# Run the Algorithm
# ============================================================

### Step 1: Create initial population

population = None  # TODO

# Prepare statistics
stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register("min", numpy.min)
stats.register("avg", numpy.mean)

# Hall of Fame stores the best individuals found
hof = tools.HallOfFame(HALL_OF_FAME_SIZE)

### Step 2: Run the GA
# Use DEAP's eaSimple algorithm
# HINT: pass population, toolbox, P_CROSSOVER, P_MUTATION, MAX_GENERATIONS, stats, hof, and verbose=True

### TODO
# population, logbook = algorithms.eaSimple(...)

# print best solution found:
best = hof.items[0]
print("-- Best Individual = ", best)
print("-- Best Fitness = ", best.fitness.values[0])
print()
print("-- Schedule = ")
nsp.printScheduleInfo(best)

In [ ]:
# ============================================================
# 8️⃣ Visualize the Results
# ============================================================

# Extract fitness statistics
### TODO: extract min and avg fitness from logbook
# minFitnessValues, meanFitnessValues = logbook.select(...)

# Plot statistics
sns.set_style("whitegrid")
plt.plot(minFitnessValues, color='red', label='Min Fitness')
plt.plot(meanFitnessValues, color='green', label='Avg Fitness')
plt.xlabel('Generation')
plt.ylabel('Fitness')
plt.title('Min and Average Fitness over Generations')
plt.legend()
plt.show()
